# The Anatomy of AI · Lecture 8 · Asking a Neuron to Draw — the notebook

Instead of asking which dots of a photo mattered, ask a neuron what picture it would love most, and let it draw: change
the *picture*, never the network, a small step uphill at a time. Naive climbing gives static that beats every real photo;
three fixes give honest pictures. Here you write the whole method from scratch and run the lecture's ladder:

1. climb the slope from a grey picture, and meet the **dead start**;
2. the naive climb: static that beats every real car;
3. the fixes: jitter, stretch and turn, waves first (Fourier), and all three together (Lucid 2017);
4. measure what each picture costs and how well it **matches the photos**;
5. draw half a layer (a mini-Microscope) and count the dogs; then **DeepDream**.

Each section names the lecture plate it follows. A finished picture depends on the GPU's arithmetic: the same seed on
two kinds of GPU gives the same kind of picture but not the same numbers (the lecture saw this too). So wherever the
lecture's number is a drawing's activation or match, the check allows a margin and says so; the claims the lecture
makes about them (zero, bigger than, smaller than) are checked exactly.

**Runs on:** free Colab with a T4 GPU (Runtime → Change runtime type → T4), in about fifteen minutes. It downloads the
model (27 MB) and the lecture's copy of the ImageNet test photos (two files of about 350 MB).

## 0. Setup
Installs (pinned), the plot style, and `check()`.

In [ ]:
# Pinned installs. Free Colab already has torch and torchvision; a machine without them gets the pinned pair.
import importlib.util, subprocess, sys
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "torchvision==0.20.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "pillow==11.1.0", "pyarrow==18.1.0", "requests==2.32.3")
pip("--no-deps", "torch-lucent==0.1.8")       # only its InceptionV1 port is used

In [ ]:
import io, json, math, os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", "a GPU" if DEV == "cuda" else "the CPU")
T0 = time.time()

# The course's look: black ink on paper, no colour except real photos.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    extra = f" ± {tol:g}" if isinstance(expected, (int, float)) and not isinstance(expected, bool) and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")

In [ ]:
LECTURE = json.loads(r'''{"neurons": ["mixed3b:379", "mixed4b:373", "mixed4c:447", "mixed4d:161", "mixed4e:55", "mixed5a:400"], "top60": {"mixed3b:379": [35460, 36425, 32907, 27182, 27389, 35506, 27923, 34708, 5074, 2052, 23690, 46967, 30348, 13243, 26379, 35753, 30512, 29342, 10917, 40372, 623, 37804, 24523, 14837, 46102, 30122, 7029, 10507, 12030, 6201, 3974, 35465, 34801, 9388, 32062, 35014, 20031, 24781, 32299, 35524, 22718, 11954, 24277, 10315, 40527, 41657, 5440, 24486, 25227, 2112, 8642, 40008, 41324, 27367, 22667, 8506, 35247, 48622, 15482, 46022], "mixed4b:373": [49888, 8038, 49176, 16612, 1915, 25468, 18767, 4495, 35787, 16834, 41919, 5457, 28204, 7843, 26496, 14645, 42365, 26125, 1770, 38955, 17987, 32471, 19045, 34974, 38948, 41153, 1177, 6449, 8012, 45788, 8766, 26353, 35236, 16089, 46640, 5344, 20828, 40074, 25158, 8194, 35220, 29221, 20753, 29200, 37158, 20318, 23095, 21001, 33841, 5881, 16944, 20825, 18394, 11416, 37828, 16336, 21984, 21813, 32893, 33520], "mixed4c:447": [25468, 26125, 5808, 35787, 8012, 38948, 49888, 41872, 41919, 1177, 3111, 44804, 7843, 2326, 20753, 7373, 18767, 10586, 10880, 31140, 26353, 32189, 22742, 46792, 26906, 41511, 14139, 16944, 21001, 31634, 42415, 48, 20257, 8194, 22512, 28204, 37882, 33520, 4495, 18394, 13516, 792, 32337, 6083, 14645, 21716, 16649, 23473, 32471, 39263, 32218, 29594, 20828, 18359, 26496, 40074, 42596, 44833, 20825, 16834], "mixed4d:161": [17684, 42000, 49562, 17044, 15960, 23146, 28554, 8075, 45284, 6904, 19433, 21291, 10747, 40513, 45581, 39985, 14867, 33668, 38836, 24769, 22573, 46682, 30226, 1075, 38312, 45911, 8969, 21720, 3051, 26568, 13470, 12390, 23126, 32427, 4056, 24470, 35060, 28405, 5637, 12649, 25489, 13216, 47045, 15362, 34284, 8602, 6588, 1882, 17807, 35761, 36630, 19454, 46641, 44571, 29696, 47298, 22117, 28644, 36895, 45474], "mixed4e:55": [38200, 7405, 23266, 38660, 22821, 29561, 39620, 31907, 8875, 33419, 27480, 10710, 42482, 15081, 11094, 8341, 5657, 18780, 12160, 33560, 8937, 11091, 23897, 14685, 15411, 795, 39453, 34493, 42977, 18366, 30051, 2071, 6083, 40237, 18380, 20480, 9842, 12338, 23679, 22917, 40744, 17139, 13672, 25607, 41569, 29854, 45945, 43671, 35164, 32909, 8635, 29360, 20469, 24499, 33796, 26411, 27225, 9502, 156, 14132], "mixed5a:400": [40660, 31314, 15051, 22956, 11574, 22537, 34324, 764, 19111, 5182, 22251, 37189, 18355, 48897, 45130, 36085, 38138, 41570, 6505, 5886, 25009, 3439, 37173, 34432, 12706, 18363, 33331, 15834, 4334, 38317, 6567, 17697, 5995, 29256, 48148, 46753, 11933, 19415, 26032, 6189, 23886, 41940, 36351, 34576, 33167, 44935, 1966, 47676, 48625, 35051, 24000, 47550, 618, 42381, 168, 15488, 22559, 29755, 31283, 25361]}, "top60_mean": {"mixed3b:379": 1102.8, "mixed4b:373": 1167.65, "mixed4c:447": 2216.517, "mixed4d:161": 574.504, "mixed4e:55": 371.054, "mixed5a:400": 224.804}, "best_spot": {"mixed3b:379": 1246.0, "mixed4b:373": 1412.0, "mixed4c:447": 2794.0, "mixed4d:161": 876.5, "mixed4e:55": 487.5, "mixed5a:400": 302.25}, "ladder": {"0_dead": {"mixed3b:379": {"act": 1104.098, "match": 0.003, "top": "wool"}, "mixed4b:373": {"act": 2508.531, "match": 0.0, "top": "window screen"}, "mixed4c:447": {"act": 0.0, "match": 0.001, "top": "paper towel"}, "mixed4d:161": {"act": 0.0, "match": 0.029, "top": "paper towel"}, "mixed4e:55": {"act": 0.0, "match": 0.007, "top": "paper towel"}, "mixed5a:400": {"act": 0.0, "match": 0.03, "top": "paper towel"}}, "1_naive": {"mixed3b:379": {"act": 1110.711, "match": 0.002, "top": "wool"}, "mixed4b:373": {"act": 2499.745, "match": 0.001, "top": "tarantula"}, "mixed4c:447": {"act": 3243.3, "match": 0.006, "top": "lionfish"}, "mixed4d:161": {"act": 2221.865, "match": 0.002, "top": "stole"}, "mixed4e:55": {"act": 1089.746, "match": 0.004, "top": "window screen"}, "mixed5a:400": {"act": 1589.888, "match": 0.009, "top": "sea urchin"}}, "2_l2_blur": {"mixed3b:379": {"act": 1045.45, "match": 0.014, "top": "vault"}, "mixed4b:373": {"act": 2435.523, "match": 0.02, "top": "half track"}, "mixed4c:447": {"act": 3434.733, "match": 0.294, "top": "jeep"}, "mixed4d:161": {"act": 1964.34, "match": 0.023, "top": "chiton"}, "mixed4e:55": {"act": 1128.495, "match": 0.098, "top": "bubble"}, "mixed5a:400": {"act": 1527.907, "match": 0.076, "top": "chest"}}, "3_jitter": {"mixed3b:379": {"act": 734.088, "match": 0.097, "top": "dome"}, "mixed4b:373": {"act": 1428.76, "match": 0.013, "top": "half track"}, "mixed4c:447": {"act": 1922.153, "match": 0.181, "top": "pickup"}, "mixed4d:161": {"act": 1255.222, "match": 0.177, "top": "vase"}, "mixed4e:55": {"act": 713.918, "match": 0.028, "top": "tennis ball"}, "mixed5a:400": {"act": 867.116, "match": 0.053, "top": "wool"}}, "4_transforms": {"mixed3b:379": {"act": 665.892, "match": 0.115, "top": "dome"}, "mixed4b:373": {"act": 1288.158, "match": 0.032, "top": "half track"}, "mixed4c:447": {"act": 1684.493, "match": 0.171, "top": "tow truck"}, "mixed4d:161": {"act": 1057.139, "match": 0.308, "top": "saltshaker"}, "mixed4e:55": {"act": 658.831, "match": 0.445, "top": "tabby"}, "mixed5a:400": {"act": 631.381, "match": 0.081, "top": "tray"}}, "5_fft_only": {"mixed3b:379": {"act": 1089.95, "match": 0.0, "top": "stole"}, "mixed4b:373": {"act": 2454.572, "match": 0.001, "top": "brain coral"}, "mixed4c:447": {"act": 3372.487, "match": 0.003, "top": "sea urchin"}, "mixed4d:161": {"act": 2134.715, "match": 0.001, "top": "stole"}, "mixed4e:55": {"act": 1182.462, "match": 0.007, "top": "tarantula"}, "mixed5a:400": {"act": 1586.379, "match": 0.002, "top": "starfish"}}, "6_lucid2017": {"mixed3b:379": {"act": 730.112, "match": 0.315, "top": "dome"}, "mixed4b:373": {"act": 1546.42, "match": 0.004, "top": "half track"}, "mixed4c:447": {"act": 2115.345, "match": 0.159, "top": "tow truck"}, "mixed4d:161": {"act": 1136.612, "match": 0.307, "top": "saltshaker"}, "mixed4e:55": {"act": 774.599, "match": 0.3, "top": "black-footed ferret"}, "mixed5a:400": {"act": 864.364, "match": 0.447, "top": "Polaroid camera"}}, "6_lucid2017_s1": {"mixed3b:379": {"act": 729.549, "match": 0.121, "top": "dome"}, "mixed4b:373": {"act": 1504.376, "match": 0.015, "top": "half track"}, "mixed4c:447": {"act": 2069.841, "match": 0.143, "top": "tow truck"}, "mixed4d:161": {"act": 1262.119, "match": 0.307, "top": "saltshaker"}, "mixed4e:55": {"act": 805.6, "match": 0.229, "top": "black-footed ferret"}, "mixed5a:400": {"act": 846.903, "match": 0.195, "top": "hard disc"}}}, "jitter": {"0": {"act": 3453.349, "match": 0.005, "top": "brain coral"}, "2": {"act": 1712.664, "match": 0.269, "top": "tow truck"}, "4": {"act": 1876.21, "match": 0.206, "top": "tow truck"}, "8": {"act": 1973.395, "match": 0.211, "top": "tow truck"}, "16": {"act": 1906.046, "match": 0.197, "top": "tow truck"}}, "dream": {"mixed3b": {"photo": "images/970/val_00001.jpeg", "before": [["ski", 0.974], ["alp", 0.022], ["shovel", 0.003]], "after": [["piggy bank", 0.322], ["spindle", 0.167], ["wool", 0.098]], "first": 963.4501, "last": 4262.6641}, "mixed4c": {"photo": "images/970/val_00001.jpeg", "before": [["ski", 0.974], ["alp", 0.022], ["shovel", 0.003]], "after": [["grey fox", 0.7], ["red fox", 0.181], ["kit fox", 0.118]], "first": 1625.8672, "last": 12877.7949}}, "dogs_4c": 56, "dogs_3b": 1, "dogs_4c_first256": 23, "dogs_3b_first64": 0, "scope_447_top": "tow truck"}''')
print(len(LECTURE['ladder']), 'rungs of the ladder in the lecture\'s record')

In [ ]:
# The photos: ImageNet's 50,000 validation photos, in the exact copy the course measured (a public 256-pixel copy,
# benjamin-paine/imagenet-1k-256x256 on Hugging Face: two parquet files of 25,000 photos, about 350 MB each, no key
# needed). Photo n here is photo n in the lecture. We read 100 photos at a time, only the ones we ask for.
import pyarrow.parquet as pq
import requests
from PIL import Image

SHARD = ("https://huggingface.co/datasets/benjamin-paine/imagenet-1k-256x256/resolve/main/data/"
         "validation-{:05d}-of-00002.parquet")
_files = {}


def _shard(s):
    path = f"val-{s}.parquet"
    if not os.path.exists(path):
        t = time.time()
        with requests.get(SHARD.format(s), stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded photo file {s} ({os.path.getsize(path) / 1e6:.0f} MB) in {time.time() - t:.0f} s")
    if s not in _files:
        _files[s] = pq.ParquetFile(path)
    return _files[s]


def rows(ids):
    """{n: (jpeg bytes, class number 0..999)} for the photos with these numbers."""
    groups = {}
    for n in ids:
        groups.setdefault((n // 25000, (n % 25000) // 100), []).append(int(n))
    out = {}
    for (s, g), ns in sorted(groups.items()):
        t = _shard(s).read_row_group(g, columns=["image", "label"])
        ims, labs = t.column("image").to_pylist(), t.column("label").to_pylist()
        for n in ns:
            out[n] = (ims[n % 100]["bytes"], int(labs[n % 100]))
    return out


def decode(b, size=224):
    """JPEG bytes (256 px square) -> the centre 224 x 224, as a 3 x 224 x 224 tensor in [0, 1]."""
    im = Image.open(io.BytesIO(b)).convert("RGB")
    w, h = im.size
    if (w, h) != (256, 256):
        s = 256 / min(w, h)
        im = im.resize((round(w * s), round(h * s)), Image.BICUBIC)
        w, h = im.size
    l, t = (w - size) // 2, (h - size) // 2
    im = im.crop((l, t, l + size, t + size))
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0).permute(2, 0, 1)

In [ ]:
# InceptionV1 (GoogLeNet, 2014), the network this lecture studies, through lucent's PyTorch port.
# One fix first: the port's local response normalization used the wrong constants (lucent pull request #51, Gorton 2024).
# We correct the installed file before importing it, as every lecture in the course does.
import importlib, importlib.util, pathlib
_spec = importlib.util.find_spec("lucent")
_f = pathlib.Path(_spec.origin).parent / "modelzoo" / "inceptionv1" / "InceptionV1.py"
_s = _f.read_text()
_old = "size=9, alpha=9.99999974738e-05, beta=0.5, k=1"
if _old in _s:
    _f.write_text(_s.replace(_old, "size=10, alpha=10 * 0.00009999999747378752, beta=0.5, k=2"))
for _p in (_f.parent / "__pycache__").glob("InceptionV1*.pyc"):
    _p.unlink()                                   # a stale cached copy would keep the old constants
importlib.invalidate_caches()

import contextlib
from lucent.modelzoo import inceptionv1
with contextlib.redirect_stderr(io.StringIO()):            # keep the download's percentage bar off the page
    net = inceptionv1(pretrained=True).eval().to(DEV)
print("InceptionV1's weights (inception5h, 27 MB): ready")
for p in net.parameters():
    p.requires_grad_(False)

# Make sure the fix took: watch the normalization during one pass.
_M = importlib.import_module("lucent.modelzoo.inceptionv1.InceptionV1")
_seen, _orig = [], _M.F.local_response_norm
_M.F.local_response_norm = lambda x, size, alpha=1e-4, beta=0.75, k=1.0: (_seen.append((size, k)), _orig(x, size, alpha, beta, k))[1]
with torch.no_grad():
    net(torch.zeros(1, 3, 224, 224, device=DEV))
_M.F.local_response_norm = _orig
check("normalization size and k, every call", sorted(set(_seen)), [(10, 2)])


def prep(x01):
    """The port takes RGB in [-117, 138], as the original TensorFlow model did."""
    return x01 * 255.0 - 117.0


class Taps:
    """Look inside the network. record=[module names] keeps their outputs in taps[name];
    edit={name: fn} replaces a module's output with fn(output) on the way through."""
    def __init__(self, record=(), edit=None):
        mods = dict(net.named_modules())
        self.out, self.h = {}, []
        for name in record:
            self.h.append(mods[name].register_forward_hook(lambda m, i, o, n=name: self.out.__setitem__(n, o)))
        for name, fn in (edit or {}).items():
            self.h.append(mods[name].register_forward_hook(lambda m, i, o, fn=fn: fn(o)))

    def __getitem__(self, name):
        return self.out[name]

    def __enter__(self):
        return self

    def __exit__(self, *a):
        for h in self.h:
            h.remove()


# InceptionV1 has 1,008 outputs in its own order; PERM puts the 1,000 classes in the standard order (the lecture's
# perm.json, which also fixes two mislabelled outputs), and NAMES gives each class its name.
PERM = torch.tensor(json.loads("[449, 450, 442, 443, 444, 445, 446, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 494, 495, 496, 497, 498, 499, 500, 501, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 475, 476, 474, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 214, 215, 217, 216, 213, 218, 647, 648, 649, 650, 651, 652, 653, 654, 655, 656, 226, 613, 614, 615, 616, 617, 618, 619, 620, 612, 422, 423, 424, 425, 427, 426, 428, 429, 430, 438, 431, 432, 433, 434, 435, 436, 437, 439, 440, 441, 6, 22, 193, 14, 173, 99, 87, 69, 116, 198, 43, 89, 200, 98, 161, 132, 32, 180, 42, 18, 207, 181, 105, 77, 189, 20, 204, 63, 31, 66, 21, 25, 45, 170, 119, 210, 107, 126, 88, 145, 59, 160, 152, 72, 33, 91, 4, 27, 113, 123, 36, 156, 109, 158, 131, 26, 71, 56, 146, 144, 125, 176, 139, 134, 90, 2, 192, 154, 150, 94, 5, 19, 191, 196, 117, 141, 68, 93, 51, 208, 184, 97, 29, 171, 124, 128, 47, 64, 211, 112, 187, 114, 86, 115, 79, 130, 140, 110, 82, 17, 177, 149, 15, 3, 41, 127, 84, 143, 133, 60, 172, 179, 118, 168, 148, 70, 197, 50, 106, 49, 151, 46, 205, 102, 28, 58, 155, 136, 202, 34, 62, 1, 159, 67, 174, 55, 10, 95, 8, 11, 201, 85, 153, 30, 190, 76, 206, 61, 163, 103, 209, 74, 35, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 224, 631, 632, 633, 634, 635, 636, 637, 638, 639, 640, 641, 642, 643, 644, 645, 646, 225, 657, 658, 188, 129, 164, 157, 13, 53, 183, 195, 101, 39, 80, 147, 78, 120, 167, 108, 162, 165, 81, 52, 9, 65, 57, 12, 121, 186, 48, 23, 182, 40, 212, 44, 16, 178, 38, 83, 104, 96, 185, 122, 73, 135, 137, 138, 203, 54, 100, 175, 142, 166, 37, 111, 92, 199, 75, 194, 24, 7, 169, 447, 451, 448, 452, 453, 456, 457, 454, 455, 547, 853, 896, 223, 345, 246, 230, 232, 677, 265, 264, 522, 688, 845, 752, 540, 847, 704, 767, 233, 907, 967, 341, 718, 916, 307, 705, 683, 518, 905, 258, 807, 908, 296, 353, 785, 909, 884, 266, 733, 991, 849, 777, 811, 933, 941, 254, 985, 835, 533, 839, 689, 252, 940, 805, 300, 706, 779, 538, 817, 716, 872, 719, 949, 851, 820, 580, 833, 887, 707, 267, 673, 591, 539, 239, 377, 836, 576, 568, 891, 749, 563, 548, 890, 929, 701, 241, 987, 342, 914, 827, 721, 902, 382, 762, 303, 335, 301, 801, 690, 695, 370, 713, 797, 979, 761, 996, 674, 698, 583, 543, 708, 243, 268, 376, 347, 910, 881, 297, 545, 778, 898, 298, 670, 756, 856, 865, 720, 313, 550, 959, 966, 523, 529, 315, 821, 667, 579, 715, 253, 897, 972, 834, 336, 799, 1000, 662, 512, 346, 262, 316, 879, 660, 808, 796, 305, 235, 279, 919, 995, 356, 309, 784, 261, 712, 934, 299, 256, 348, 671, 757, 280, 971, 567, 955, 275, 792, 276, 236, 337, 911, 227, 684, 725, 703, 517, 581, 895, 249, 375, 840, 505, 551, 750, 573, 350, 344, 554, 369, 787, 696, 730, 562, 802, 924, 293, 525, 980, 659, 592, 748, 269, 961, 963, 294, 560, 770, 773, 582, 918, 892, 814, 228, 374, 988, 371, 687, 238, 546, 270, 244, 867, 973, 894, 508, 536, 697, 532, 818, 917, 782, 977, 763, 338, 339, 781, 984, 598, 922, 946, 302, 717, 509, 661, 866, 875, 920, 880, 271, 251, 871, 829, 290, 272, 764, 686, 869, 277, 824, 854, 691, 852, 260, 255, 728, 511, 599, 284, 588, 585, 595, 755, 915, 552, 699, 354, 351, 520, 513, 333, 870, 937, 295, 506, 921, 826, 564, 584, 504, 759, 685, 352, 877, 942, 994, 306, 527, 257, 679, 843, 732, 842, 850, 883, 783, 789, 575, 926, 722, 281, 596, 931, 901, 888, 841, 565, 245, 983, 372, 693, 964, 731, 381, 378, 857, 923, 285, 855, 314, 788, 838, 566, 373, 769, 556, 702, 542, 534, 572, 846, 939, 862, 976, 273, 860, 571, 863, 537, 927, 286, 570, 965, 668, 578, 694, 219, 541, 310, 663, 997, 876, 519, 760, 753, 586, 952, 751, 938, 355, 809, 521, 962, 221, 729, 510, 587, 379, 589, 559, 800, 709, 832, 950, 259, 380, 868, 747, 590, 776, 943, 549, 726, 557, 507, 288, 278, 960, 222, 986, 577, 925, 822, 858, 515, 234, 676, 237, 600, 775, 274, 593, 804, 263, 680, 340, 530, 998, 724, 528, 516, 514, 287, 957, 311, 692, 247, 794, 526, 574, 535, 810, 681, 828, 837, 945, 569, 561, 531, 304, 250, 978, 675, 786, 944, 970, 989, 903, 758, 555, 308, 780, 664, 710, 312, 594, 700, 282, 711, 289, 283, 766, 825, 291, 242, 597, 678, 882, 349, 765, 727, 544, 220, 292, 334, 666, 874, 990, 969, 558, 790, 682, 343, 936, 665, 524, 928, 317, 231, 906, 669, 958, 819, 795, 772, 502, 899, 912, 904, 935, 831, 503, 672, 951, 815, 723, 248, 240, 714, 553, 930, 791, 932, 861, 774, 803, 754, 813, 844, 771, 793, 974, 968, 873, 768, 975, 993, 885, 734, 736, 737, 738, 739, 740, 741, 742, 743, 744, 735, 745, 746, 318, 229, 319, 320, 321, 322, 323, 324, 325, 326, 816, 999, 953, 864, 806, 948, 830, 900, 798, 947, 859, 823, 361, 992, 359, 365, 368, 366, 363, 364, 367, 360, 362, 954, 848, 982, 330, 357, 358, 331, 327, 328, 332, 886, 913, 956, 893, 878, 812, 981, 329, 889]"), device=DEV)
from torchvision.models import GoogLeNet_Weights
NAMES = GoogLeNet_Weights.IMAGENET1K_V1.meta["categories"]

In [ ]:
# lucent's ReLUs pass gradients backwards with a trick for drawing ("redirect", section 1). We keep it, and can switch
# to the plain, true gradient for honest measurements.
for _m in net.modules():
    if type(_m).__name__ == "RedirectedReluLayer":
        _m._redirect = _m.forward


def set_relu(mode):
    for m in net.modules():
        if hasattr(m, "_redirect"):
            m.forward = m._redirect if mode == "redirect" else (lambda x: F.relu(x))

set_relu("plain")
INKMAP = plt.matplotlib.colors.LinearSegmentedColormap.from_list("ink", [PAPER, INK])
DOG = slice(151, 269)                     # the 118 dog breeds of ImageNet

## 1. The method, from scratch (Plates I–II)
Start with a grey picture, 128 × 128, with a little noise. Ask, for every pixel at once: brighter or darker, which way
makes the neuron fire more? That is Lecture 7's slope, aimed at one neuron (its **activation**: the mean of its map).
Move every pixel a small step uphill, with the Adam optimiser, 512 times. This is **gradient ascent on the image**.

Everything else below is a switchable fix. The picture can be stored as its pixels, or as its **Fourier waves** (slow
ones get bigger steps, and colours are mixed the way photos mix them). Before each step the picture can be moved,
stretched and turned a little (**transforms**). One more knob, the ReLU's backward rule, comes up in a moment.

In [ ]:
import math
LAYER_ORDER = ["conv2d0", "conv2d1", "conv2d2", "mixed3a", "mixed3b", "mixed4a", "mixed4b", "mixed4c", "mixed4d",
               "mixed4e", "mixed5a", "mixed5b"]
COLOR = np.array([[0.26, 0.09, 0.02], [0.27, 0.00, -0.05], [0.27, -0.09, 0.03]], np.float32)   # Lucid's colour mixing
COLOR = COLOR / np.max(np.linalg.norm(COLOR, axis=0))


class Picture:
    """B pictures we optimise. 'pixel': the pixels themselves. 'fft': their Fourier waves, slow waves scaled up."""
    def __init__(self, B, size, param="fft", sd=0.01, seed=0):
        g = torch.Generator().manual_seed(seed)
        self.param, self.size = param, size
        if param == "fft":
            fy, fx = np.fft.fftfreq(size)[:, None], np.fft.fftfreq(size)[: size // 2 + 1]
            freq = np.sqrt(fx * fx + fy * fy)
            self.t = (torch.randn(B, 3, *freq.shape, 2, generator=g) * sd).to(DEV).requires_grad_(True)
            self.scale = torch.tensor(1.0 / np.maximum(freq, 1.0 / size), dtype=torch.float32, device=DEV)[None, None, ..., None]
            self.C = torch.tensor(COLOR, device=DEV)
        else:                                      # grey, plus a little noise, kept in [0, 1]
            self.t = (0.5 + torch.randn(B, 3, size, size, generator=g) * sd).to(DEV).requires_grad_(True)

    def get(self):
        if self.param == "pixel":
            return self.t
        s = self.scale * self.t
        x = torch.fft.irfftn(torch.view_as_complex(s.contiguous()), s=(self.size, self.size), norm="ortho") / 4.0
        return torch.sigmoid(torch.einsum("bchw,dc->bdhw", x, self.C))

    def after_step(self):
        if self.param == "pixel":
            with torch.no_grad():
                self.t.clamp_(0, 1)


def pad(x, p):
    return F.pad(x, (p, p, p, p), value=0.5)

def jitter(x, d):
    dx, dy = np.random.randint(d), np.random.randint(d)
    return x[:, :, dy:x.shape[2] - d + dy, dx:x.shape[3] - d + dx]

def rscale(x, scales):
    s = float(np.random.choice(scales))
    return F.interpolate(x, scale_factor=s, mode="bilinear", align_corners=False) if s != 1 else x

def rrotate(x, angles):
    a = math.radians(float(np.random.choice(angles)))
    th = torch.tensor([[math.cos(a), -math.sin(a), 0], [math.sin(a), math.cos(a), 0]], dtype=x.dtype, device=x.device)
    grid = F.affine_grid(th[None].expand(x.shape[0], 2, 3), x.shape, align_corners=False)
    return F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)

def standard(x):                                   # Lucid 2017's transforms: pad, jitter, stretch, turn, jitter
    x = jitter(pad(x, 12), 8)
    x = rscale(x, [1 + (i - 5) / 50.0 for i in range(11)])
    x = rrotate(x, list(range(-10, 11)) + 5 * [0])
    return jitter(x, 4)

TF = {"none": lambda x: x, "jitter": lambda x: jitter(pad(x, 8), 8), "standard": standard}


class _Stop(Exception):
    pass


def capture(layer, x):
    """Run the network on x and return `layer`'s maps, stopping there (nothing above it is needed)."""
    out = {}
    def hook(m, i, o):
        out["a"] = o
        raise _Stop
    h = dict(net.named_modules())[layer].register_forward_hook(hook)
    try:
        net(prep(x))
    except _Stop:
        pass
    finally:
        h.remove()
    return out["a"]


def render(layer, ch, param="fft", tf="standard", relu="redirect", size=128, steps=512, lr=0.05, seed=0, snaps=()):
    """Draw neuron layer:ch. Returns the picture (1, 3, size, size) and any snapshots {step: picture}."""
    np.random.seed(seed); torch.manual_seed(seed)
    set_relu(relu)
    pic = Picture(1, size, param, seed=seed)
    opt = torch.optim.Adam([pic.t], lr=lr)
    kept = {}
    for i in range(steps):
        if i in snaps:
            kept[i] = pic.get().detach().cpu()
        act = capture(layer, TF[tf](pic.get()))[0, ch].mean()
        opt.zero_grad(); (-act).backward(); opt.step()
        pic.after_step()
    set_relu("plain")
    return pic.get().detach(), kept

**Reading a finished picture honestly.** With no transforms and the true gradient rule: the neuron's activation on it,
what the whole network calls it (top class), and how well it **matches the photos**: compare the network's 1,000
chances for the picture with its average chances over the neuron's 60 strongest real test photos (cosine: 1 = the
network reads the picture exactly as it reads those photos, 0 = nothing in common).

In [ ]:
def chances(x):
    with torch.no_grad(), Taps(["softmax2_pre_activation_matmul"]) as T:
        net(prep(F.interpolate(x, size=224, mode="bilinear", align_corners=False)))
    return T["softmax2_pre_activation_matmul"][:, PERM].softmax(1)


# each neuron's 60 strongest test photos (from Lecture 6's pass over all 50,000), and how the network reads them
NEURONS = [(k.split(":")[0], int(k.split(":")[1])) for k in LECTURE["neurons"]]
want = sorted({i for k in LECTURE["neurons"] for i in LECTURE["top60"][k]})
photos = rows(want)
profile = {}
for k in LECTURE["neurons"]:
    L, c = k.split(":")
    X = torch.stack([decode(photos[i][0]) for i in LECTURE["top60"][k]]).to(DEV)
    with torch.no_grad(), Taps(["softmax2_pre_activation_matmul", L]) as T:
        net(prep(X))
    profile[k] = T["softmax2_pre_activation_matmul"][:, PERM].softmax(1).mean(0)
    best = T[L][:, int(c)].flatten(1).max(1).values
    check(f"{k}: its 60 strongest photos, mean of their strongest spots", best.mean().item(), LECTURE["top60_mean"][k],
          tol=0.003 * LECTURE["top60_mean"][k])
del photos


def read(layer, ch, x):
    with torch.no_grad():
        act = capture(layer, x)[0, ch].mean().item()
    p = chances(x)[0]
    k = f"{layer}:{ch}"
    match = float(p @ profile[k] / (p.norm() * profile[k].norm()))
    return {"act": act, "match": match, "top": NAMES[int(p.argmax())], "p_top": float(p.max()), "p_dog": float(p[DOG].sum())}

## 2. Nothing happens: the dead start (Plate III)
Climb with the true gradient. On a grey picture the deep neurons are silent: below zero, on the flat side of Lecture 1's
gate, where the slope is exactly 0. No slope, no step, and after 512 steps they are still exactly 0. The curve neuron
3b:379, which starts above zero, climbs.

In [ ]:
ladder = {}
t = time.time()
ladder["0_dead"] = {}
for L, c in NEURONS:
    x, _ = render(L, c, param="pixel", tf="none", relu="plain")
    ladder["0_dead"][f"{L}:{c}"] = (x.cpu(), read(L, c, x))
print(f"six neurons, 512 steps each, in {time.time() - t:.0f} s")
for k, (_, r) in ladder["0_dead"].items():
    want = LECTURE["ladder"]["0_dead"][k]["act"]
    if want == 0:
        check(f"{k} after 512 true-gradient steps: still exactly", r["act"], 0.0)
    else:
        check(f"{k} climbs anyway (it starts above zero)", r["act"], want, tol=0.05 * want)

**The fix (lucent's redirected ReLU):** on the way back only, if a ReLU would block every slope, let through the part
that would push its input up. The forward pass is unchanged, so every reading is still honest.

## 3. Static, and the fixes (Plates IV–IX)
Now draw all six of the lecture's neurons, rung by rung:

| rung | picture stored as | transforms | |
|---|---|---|---|
| 1 naive | pixels | none | Plate IV: static |
| 2 + L2 and blur | pixels, pulled to grey and blurred | none | the 2009/2015 fix |
| 3 + jitter | pixels | moved a few pixels | Plate VI |
| 4 + transforms | pixels | moved, stretched, turned | Plate VII |
| 5 Fourier only | waves | none | Plate VIII |
| 6 Lucid 2017 | waves | moved, stretched, turned | Plate IX: all three |

In [ ]:
def l2_blur_render(layer, ch, steps=512, seed=0):
    """Rung 2: naive pixels, but every 4 steps blur the picture, and pull it toward grey in the objective."""
    np.random.seed(seed); torch.manual_seed(seed)
    set_relu("redirect")
    pic = Picture(1, 128, "pixel", seed=seed)
    opt = torch.optim.Adam([pic.t], lr=0.05)
    k = torch.exp(-torch.arange(-2, 3, device=DEV, dtype=torch.float32) ** 2 / 0.5)
    k = k / k.sum()
    def blur(z):
        z = F.conv2d(F.pad(z, (2, 2, 0, 0), mode="reflect"), k.view(1, 1, 1, 5).repeat(3, 1, 1, 1), groups=3)
        return F.conv2d(F.pad(z, (0, 0, 2, 2), mode="reflect"), k.view(1, 1, 5, 1).repeat(3, 1, 1, 1), groups=3)
    for i in range(steps):
        x = pic.get()
        if i % 4 == 0:
            with torch.no_grad():
                pic.t.copy_(blur(pic.t))
        act = capture(layer, x)[0, ch].mean()
        loss = -act + 0.5 * ((x - 0.5) ** 2).mean() * act.detach().abs().clamp(min=1.0)
        opt.zero_grad(); loss.backward(); opt.step()
        pic.after_step()
    set_relu("plain")
    return pic.get().detach()


SPEC = {"1_naive": ("pixel", "none"), "3_jitter": ("pixel", "jitter"), "4_transforms": ("pixel", "standard"),
        "5_fft_only": ("fft", "none"), "6_lucid2017": ("fft", "standard")}
snaps = {}
t = time.time()
for rung in ("1_naive", "2_l2_blur", "3_jitter", "4_transforms", "5_fft_only", "6_lucid2017"):
    ladder[rung] = {}
    for L, c in NEURONS:
        if rung == "2_l2_blur":
            x = l2_blur_render(L, c)
        else:
            param, tf = SPEC[rung]
            want_snaps = (0, 4, 16, 64, 256) if (rung in ("1_naive", "6_lucid2017") and (L, c) == ("mixed4c", 447)) else ()
            x, kept = render(L, c, param=param, tf=tf, snaps=want_snaps)
            if kept:
                snaps[rung] = kept
        ladder[rung][f"{L}:{c}"] = (x.cpu(), read(L, c, x))
print(f"36 pictures in {time.time() - t:.0f} s")
print(f"{'rung':14s}" + "".join(f"{k:>14s}" for k in LECTURE["neurons"]))
for rung in ladder:
    print(f"{rung:14s}" + "".join(f"{ladder[rung][k][1]['act']:14,.0f}" for k in LECTURE["neurons"]))
print("\nmatches the photos:")
for rung in ladder:
    print(f"{rung:14s}" + "".join(f"{ladder[rung][k][1]['match']:14.2f}" for k in LECTURE["neurons"]))

**Static beats every real car (Plates IV–V).** The naive picture drives the car neuron's *average over the whole
picture* above its single strongest spot on the best of 50,000 real photos, and the network reads it as something else
entirely. Nothing kept the picture photo-like: it packed fine patterns exactly where the first layer's tiny edge
detectors look.

In [ ]:
naive = ladder["1_naive"]["mixed4c:447"][1]
print(f"naive car picture: activation {naive['act']:,.0f}; the network calls it {naive['top']} ({naive['p_top']:.0%}); "
      f"match {naive['match']:.3f}")
check("naive car picture: activation (a GPU run; the lecture's run gave 3,243)", naive["act"], LECTURE["ladder"]["1_naive"]["mixed4c:447"]["act"],
      tol=0.15 * LECTURE["ladder"]["1_naive"]["mixed4c:447"]["act"])
check("it beats the strongest spot on the best real car photo", naive["act"] > LECTURE["best_spot"]["mixed4c:447"], True)
check("yet it matches the car photos hardly at all (under 0.03)", naive["match"] < 0.03, True)
for k in LECTURE["neurons"]:
    check(f"{k}: naive picture beats its 60 best photos' strongest spots", ladder["1_naive"][k][1]["act"] > LECTURE["top60_mean"][k], True)

**The fixes (Plates VI–IX), and what meaning costs (Plate X).** The transforms keep only patterns that work wherever
they land; Fourier alone climbs highest but stays static; together (Lucid 2017) they give honest pictures, and those
fire the neuron *less* than the static did, while matching the real photos far better.

In [ ]:
mean_match = {r: float(np.mean([ladder[r][k][1]["match"] for k in LECTURE["neurons"]])) for r in ladder}
lecture_mean = {r: float(np.mean([LECTURE["ladder"][r][k]["match"] for k in LECTURE["neurons"]])) for r in ladder}
for r in ladder:
    print(f"{r:14s} mean match {mean_match[r]:.3f}   (lecture {lecture_mean[r]:.3f})")
lucid = ladder["6_lucid2017"]["mixed4c:447"][1]
print(f"Lucid 2017 car: activation {lucid['act']:,.0f}, called {lucid['top']} ({lucid['p_top']:.0%}), match {lucid['match']:.2f}")
check("Lucid 2017 car: activation (lecture 2,115)", lucid["act"], LECTURE["ladder"]["6_lucid2017"]["mixed4c:447"]["act"],
      tol=0.15 * LECTURE["ladder"]["6_lucid2017"]["mixed4c:447"]["act"])
check("meaning costs activation: Lucid 2017 car < naive car", lucid["act"] < naive["act"], True)
check("Lucid 2017 car matches the car photos better than static (by over 0.1)", lucid["match"] - naive["match"] > 0.1, True)
check("mean match, naive (under 0.02)", mean_match["1_naive"] < 0.02, True)
check("mean match, Fourier alone (under 0.02)", mean_match["5_fft_only"] < 0.02, True)
s1 = float(np.mean([LECTURE["ladder"]["6_lucid2017_s1"][k]["match"] for k in LECTURE["neurons"]]))
lo, hi = sorted([lecture_mean["6_lucid2017"], s1])
print(f"the lecture's two Lucid 2017 seeds: mean match {lecture_mean['6_lucid2017']:.3f} and {s1:.3f}")
check("mean match, Lucid 2017 (inside the lecture's two seeds, give or take 0.05)", mean_match["6_lucid2017"], (lo + hi) / 2,
      tol=(hi - lo) / 2 + 0.05)
check("Lucid 2017 matches the photos over ten times better than static", mean_match["6_lucid2017"] > 10 * max(mean_match["1_naive"], 0.001), True)
check("Fourier alone climbs higher than Lucid 2017 on the car", ladder["5_fft_only"]["mixed4c:447"][1]["act"] > lucid["act"], True)

fig, axs = plt.subplots(6, 6, figsize=(10, 10.6))
for i, rung in enumerate(("1_naive", "2_l2_blur", "3_jitter", "4_transforms", "5_fft_only", "6_lucid2017")):
    for j, k in enumerate(LECTURE["neurons"]):
        x, r = ladder[rung][k]
        axs[i, j].imshow(x[0].permute(1, 2, 0)); axs[i, j].set_xticks([]); axs[i, j].set_yticks([])
        axs[i, j].set_xlabel(r["top"][:16], fontsize=7)
        if i == 0:
            axs[i, j].set_title(k, fontsize=9)
        if j == 0:
            axs[i, j].set_ylabel(rung, fontsize=8)
fig.suptitle("the ladder: six neurons, six ways of drawing them (under each: what the network calls it)", fontsize=10)
plt.show()

fig, axs = plt.subplots(2, 5, figsize=(10, 4.4))
for row, rung in zip(axs, ("1_naive", "6_lucid2017")):
    for ax, (s, x) in zip(row, sorted(snaps[rung].items())):
        ax.imshow(x[0].permute(1, 2, 0)); ax.set_title(f"{rung}, step {s}", fontsize=8); ax.axis("off")
plt.show()

## 4. How much jitter? (Plate VI)
The car neuron drawn in pixels, with the picture padded by d pixels and cut back at a random place before each step.

In [ ]:
sweep = {}
for d in (0, 2, 4, 8, 16):
    TF[f"jit{d}"] = (lambda x, d=d: jitter(pad(x, d), d)) if d else (lambda x: x)
    x, _ = render("mixed4c", 447, param="pixel", tf=f"jit{d}")
    sweep[d] = read("mixed4c", 447, x)
    print(f"jitter {d:2d} px: activation {sweep[d]['act']:6,.0f}   called {sweep[d]['top']:16s} match {sweep[d]['match']:.2f}"
          f"   (lecture: {LECTURE['jitter'][str(d)]['act']:,.0f}, {LECTURE['jitter'][str(d)]['top']}, {LECTURE['jitter'][str(d)]['match']:.2f})")
check("no jitter: static (match under 0.03)", sweep[0]["match"] < 0.03, True)
for d in (2, 4, 8, 16):
    check(f"jitter {d} px: a vehicle that matches the car photos (match over 0.1)", sweep[d]["match"] > 0.1, True)

## 5. Falling into dogs: a mini-Microscope (Plates XIII and XV)
The lecture drew *every* neuron of mixed4c, 512 pictures, with the Lucid 2017 settings, and asked the network what each
picture is. To stay inside free Colab's time we draw the first half, neurons 0–255, 64 at a time (the car neuron 447 is
drawn on its own). How many does the network read as a dog (more than half its chance on the 118 dog breeds)?

In [ ]:
def render_many(layer, chans, seed=0, steps=512):
    """Lucid 2017 settings for many neurons of one layer at once, one picture each."""
    np.random.seed(seed); torch.manual_seed(seed)
    set_relu("redirect")
    pic = Picture(len(chans), 128, "fft", seed=seed)
    opt = torch.optim.Adam([pic.t], lr=0.05)
    idx = torch.tensor(chans, device=DEV)
    for i in range(steps):
        a = capture(layer, standard(pic.get()))
        obj = a[torch.arange(len(chans)), idx].mean((1, 2))
        opt.zero_grad(); (-obj.sum()).backward(); opt.step()
    set_relu("plain")
    return pic.get().detach()


t = time.time()
scope, dogs = [], []
for s in range(0, 256, 64):
    x = render_many("mixed4c", list(range(s, s + 64)))
    p = torch.cat([chances(x[i:i + 32]) for i in range(0, 64, 32)])
    dogs.append(p[:, DOG].sum(1).cpu()); scope.append(x.cpu())
scope, dogs = torch.cat(scope), torch.cat(dogs)
print(f"256 pictures in {time.time() - t:.0f} s; read as a dog: {int((dogs > 0.5).sum())} "
      f"(the lecture: {LECTURE['dogs_4c_first256']} of these 256, {LECTURE['dogs_4c']} of all 512)")
check("mixed4c neurons 0-255 drawn as dogs (a run-to-run margin of 6)", int((dogs > 0.5).sum()), LECTURE["dogs_4c_first256"], tol=6)
car_pic = render_many("mixed4c", [447])
p447 = chances(car_pic)[0]
print(f"4c:447 in the Microscope: {NAMES[int(p447.argmax())]} ({p447.max():.0%})")

show = [0, 1, 2] + [int(i) for i in torch.argsort(-dogs)[:8]]
fig, axs = plt.subplots(2, 6, figsize=(10, 3.8))
axs.flat[0].imshow(car_pic[0].permute(1, 2, 0).cpu()); axs.flat[0].set_title(f"4c:447  {NAMES[int(p447.argmax())]}", fontsize=8)
axs.flat[0].axis("off")
for ax, c in zip(list(axs.flat)[1:], show):
    ax.imshow(scope[c].permute(1, 2, 0)); ax.set_title(f"4c:{c}  dog {dogs[c]:.0%}", fontsize=8); ax.axis("off")
plt.show()

## 6. DeepDream (Plate XIV)
The same climb on a real photograph: push a whole layer to see more of whatever it already sees (maximise the mean of
its squared maps), with the picture rolled a little each step, at four sizes from small to large (octaves), 20 steps
each. The lecture's photo is the first test photo of an alp: a mountain with two small skiers.

In [ ]:
import pyarrow.parquet as pq
labels = np.concatenate([pq.ParquetFile(f"val-{s}.parquet").read(columns=["label"]).column("label").to_numpy() for s in (0, 1)])
alp_n = int(np.where(labels == NAMES.index("alp"))[0][0])
check("the first alp test photo", f"images/970/val_{alp_n:05d}.jpeg", LECTURE["dream"]["mixed4c"]["photo"])
alp = Image.open(io.BytesIO(rows([alp_n])[alp_n][0])).convert("RGB").resize((384, 384), Image.BICUBIC)
base = torch.from_numpy(np.asarray(alp, dtype=np.float32) / 255).permute(2, 0, 1)[None].to(DEV)


def dream(layer, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    octaves = [base]
    for _ in range(3):
        octaves.append(F.interpolate(octaves[-1], scale_factor=1 / 1.4, mode="bilinear", align_corners=False))
    detail, objs = torch.zeros_like(octaves[-1]), []
    for ob in reversed(octaves):
        detail = F.interpolate(detail, size=ob.shape[-2:], mode="bilinear", align_corners=False)
        x = (ob + detail).clone()
        for _ in range(20):
            sx, sy = np.random.randint(-16, 17, 2)
            xr = torch.roll(x, (int(sy), int(sx)), (2, 3)).requires_grad_(True)
            obj = 0.5 * (capture(layer, xr) ** 2).mean()
            g, = torch.autograd.grad(obj, xr)
            with torch.no_grad():
                x = torch.roll(xr + 0.01 * g / (g.abs().mean() + 1e-8), (-int(sy), -int(sx)), (2, 3)).clamp(0, 1)
            objs.append(obj.item())
        detail = x - ob
    return x, objs


x4c, objs = dream("mixed4c")
before, after = chances(base)[0], chances(x4c)[0]
print("before:", [(NAMES[int(i)], round(float(before[i]), 3)) for i in before.topk(3).indices])
print("after: ", [(NAMES[int(i)], round(float(after[i]), 3)) for i in after.topk(3).indices])
check("before the dream, the network says", NAMES[int(before.argmax())], LECTURE["dream"]["mixed4c"]["before"][0][0])
check("how sure", float(before.max()), LECTURE["dream"]["mixed4c"]["before"][0][1], tol=0.005)
check("the layer's objective, first step", objs[0], LECTURE["dream"]["mixed4c"]["first"], tol=0.01 * LECTURE["dream"]["mixed4c"]["first"])
foxes = sum(float(after[NAMES.index(n)]) for n in ("grey fox", "red fox", "kit fox", "Arctic fox"))
print(f"after 80 steps: the foxes together {foxes:.0%}; the 118 dog breeds {after[DOG].sum():.0%}")
check("after the dream: fox-like animals take most of the chance (lecture: grey, red and kit fox 99.9%)", foxes > 0.5, True)
fig, axs = plt.subplots(1, 2, figsize=(7, 3.6))
for ax, x, t_ in zip(axs, (base, x4c), ("the alp photo", "dreamt on mixed4c")):
    ax.imshow(x[0].permute(1, 2, 0).cpu()); ax.set_title(t_, fontsize=10); ax.axis("off")
plt.show()

## Your turn
1. **Another seed.** Draw 5a:400 (the essay's box camera) with the Lucid 2017 settings and `seed=1`. Same picture? Same
   name? *Hint: `render("mixed5a", 400, seed=1)`, then `read`.*
2. **The middle layer's dogs.** Draw mixed3b's first 64 neurons (`render_many("mixed3b", list(range(64)))`) and count the
   dog pictures. *Hint: the lecture found 1 in all 480.*
3. **L2 and blur on the car.** Rung 2 is already in `ladder`. How does its car compare with the naive one and with
   Lucid 2017? *Hint: activation, name and match.*
4. **Dream a shallower layer.** DeepDream the alp on mixed3b. What does the network see now? *Hint: `dream("mixed3b")`.*

In [ ]:
# Your turn 1
# TODO

In [ ]:
# Your turn 2
# TODO

In [ ]:
# Your turn 3
# TODO

In [ ]:
# Your turn 4
# TODO

## Solutions

In [ ]:
# 1. Another seed
x, _ = render("mixed5a", 400, seed=1)
r = read("mixed5a", 400, x)
print(f"5a:400, seed 1: activation {r['act']:,.0f}, called {r['top']}, match {r['match']:.2f}")
want = LECTURE["ladder"]["6_lucid2017_s1"]["mixed5a:400"]
check("5a:400 seed 1: activation (lecture 847)", r["act"], want["act"], tol=0.15 * want["act"])
fig, axs = plt.subplots(1, 2, figsize=(4.4, 2.4))
axs[0].imshow(ladder["6_lucid2017"]["mixed5a:400"][0][0].permute(1, 2, 0)); axs[0].set_title("seed 0", fontsize=9)
axs[1].imshow(x[0].permute(1, 2, 0).cpu()); axs[1].set_title("seed 1", fontsize=9)
for a in axs:
    a.axis("off")
plt.show()
print("The same kind of boxy device; its name can change. Show the real photos beside every picture.")

In [ ]:
# 2. The middle layer's dogs
x3 = render_many("mixed3b", list(range(64)))
d3 = torch.cat([chances(x3[i:i + 32])[:, DOG].sum(1).cpu() for i in range(0, 64, 32)])
print(f"mixed3b, neurons 0-63: {int((d3 > 0.5).sum())} read as dogs (mixed4c: {int((dogs > 0.5).sum())} of 256)")
check("mixed3b's first 64 pictures read as dogs (lecture 0 of these 64; margin 1)", int((d3 > 0.5).sum()), LECTURE["dogs_3b_first64"], tol=1)

In [ ]:
# 3. L2 and blur on the car
for rung in ("1_naive", "2_l2_blur", "6_lucid2017"):
    r = ladder[rung]["mixed4c:447"][1]
    print(f"{rung:12s} activation {r['act']:6,.0f}   called {r['top']:14s} match {r['match']:.2f}")
want = LECTURE["ladder"]["2_l2_blur"]["mixed4c:447"]
check("L2 + blur car: activation (lecture 3,435)", ladder["2_l2_blur"]["mixed4c:447"][1]["act"], want["act"], tol=0.15 * want["act"])

In [ ]:
# 4. Dream a shallower layer
x3b, objs3 = dream("mixed3b")
after3 = chances(x3b)[0]
print("mixed3b dream:", [(NAMES[int(i)], round(float(after3[i]), 3)) for i in after3.topk(3).indices],
      f"  (lecture: {LECTURE['dream']['mixed3b']['after']})")
check("mixed3b dream: the objective at the first step", objs3[0], LECTURE["dream"]["mixed3b"]["first"], tol=0.01 * LECTURE["dream"]["mixed3b"]["first"])
check("mixed3b dream: no fox-like animals (under 10%)",
      sum(float(after3[NAMES.index(n)]) for n in ("grey fox", "red fox", "kit fox", "Arctic fox")) < 0.1, True)
fig, ax = plt.subplots(figsize=(3.4, 3.4)); ax.imshow(x3b[0].permute(1, 2, 0).cpu()); ax.axis("off")
ax.set_title("dreamt on mixed3b: textures", fontsize=10); plt.show()
print(f"notebook ran in {time.time() - T0:.0f} s")